## AutoShop Multi-Agent mit OpenAI Agents SDK

Dieses Notebook basiert auf dem [AutoShop MCP-Server](../05-mcp/30-autoshop-mcp-server.ipynb), der zuerst ausgeführt werden muss.

Gegenüber der bisherigen Variante wird der Agenten- und MCP-Loop nicht mehr manuell mit der Responses API implementiert. Stattdessen verwendet dieses Notebook das **OpenAI Agents SDK**:

1. `MCPServerStreamableHttp` bindet den bestehenden AutoShop-MCP-Server direkt ein.
2. Ein interner `Agent` sucht ausschliesslich über freigegebene MCP-Tools nach Fahrzeugen.
3. Ein externer `Agent` ergänzt allgemeine Informationen ohne MCP-Zugriff.
4. Ein Offert-Agent erstellt aus beiden Resultaten die Markdown-Offerte.
5. `Runner.run()` führt die einzelnen Agenten aus.

`order_delete` bleibt bewusst gesperrt.


---

### Installation

Das Agents SDK wird über das Paket `openai-agents` installiert. Das MCP-Paket wird explizit in einem vom Agents SDK unterstützten Bereich gehalten.


In [ ]:
%pip install -q -U openai openai-agents "mcp>=1.19,<3"

### Umgebung und Verbindung

Die bestehende `~/data/env.py` wird weiterverwendet. Erwartet werden mindestens:

- `OPENAI_API_KEY`
- `AI_MODEL`
- `AI_BASE_URL` (optional bzw. dein bisheriger OpenAI-kompatibler Endpoint)

Der MCP-Endpoint wird wie bisher aus Kubernetes ermittelt.


In [ ]:
%run ~/data/env.py

import subprocess
from openai import AsyncOpenAI

from agents import (
    Agent,
    Runner,
    set_default_openai_client,
    set_tracing_disabled,
)
from agents.mcp import (
    MCPServerStreamableHttp,
    create_static_tool_filter,
)


def get_server_url():
    server_ip = subprocess.check_output(
        "cat ~/data/server-ip",
        shell=True,
        text=True,
    ).strip()

    node_port = subprocess.check_output(
        "kubectl get service --namespace ms-mcp autoshop-mcp-server "
        "-o=jsonpath='{ .spec.ports[0].nodePort }'",
        shell=True,
        text=True,
    ).strip()

    return f"http://{server_ip}:{node_port}/mcp"


MCP_URL = get_server_url()

# Das Agents SDK verwendet einen asynchronen OpenAI-Client.
openai_client = AsyncOpenAI(
    api_key=OPENAI_API_KEY,
    base_url=AI_BASE_URL,
)
set_default_openai_client(openai_client, use_for_tracing=False)

# Für OpenAI-kompatible Endpoints ist Tracing nicht zwingend verfügbar.
# Bei direkter Nutzung der OpenAI Platform kann diese Zeile entfernt werden,
# wenn OpenAI Tracing verwendet werden soll.
set_tracing_disabled(True)

print(f"MCP URL: {MCP_URL}")
print(f"Model:   {AI_MODEL}")


---

### Read-only MCP-Toolset

Das Agents SDK kann MCP-Tools direkt filtern. Damit muss keine eigene MCP→OpenAI-Tool-Konvertierung mehr implementiert werden.


In [ ]:
READ_ONLY_TOOLS = [
    "catalog_list_items",
    "catalog_get_item",
    "customer_list_items",
    "customer_get_item",
    "order_list_items",
    "order_get_item",
]

read_only_filter = create_static_tool_filter(
    allowed_tool_names=READ_ONLY_TOOLS,
)


### MCP-Server für den Agenten

`MCPServerStreamableHttp` verwaltet die MCP-Verbindung. `cache_tools_list=True` verhindert unnötige `list_tools()`-Roundtrips, solange sich das Toolset während eines Runs nicht verändert.


In [ ]:
autoshop_mcp = MCPServerStreamableHttp(
    name="AutoShop MCP",
    params={
        "url": MCP_URL,
        "timeout": 30,
    },
    cache_tools_list=True,
    tool_filter=read_only_filter,
    max_retry_attempts=2,
)


---

## Agenten

### 1. Interner AutoShop-Agent

Dieser Agent besitzt als einziger Zugriff auf den AutoShop-MCP-Server. Fahrzeugdaten, IDs und Preise müssen aus MCP stammen.


In [ ]:
internal_car_agent = Agent(
    name="AutoShop Internal Car Agent",
    model=AI_MODEL,
    instructions="""
Du bist ein Auto-Verkaufsagent für den AutoShop.

Aufgabe:
1. Analysiere den Kundenwunsch.
2. Verwende die verfügbaren AutoShop-MCP-Tools.
3. Liste passende Fahrzeuge aus dem Catalog auf.
4. Verwende ausschliesslich Preise aus dem Catalog.
5. Wähle das am besten passende Fahrzeug aus.
6. Begründe die Auswahl kurz.
7. Gib die Antwort strukturiert aus.

Regeln:
- Erfinde keine Fahrzeuge, IDs, Preise oder Verfügbarkeiten.
- Shop-Daten müssen über MCP beschafft werden.
- Wenn Eigenschaften fehlen, verwende nur die tatsächlich gelieferten Catalog-Daten.
- Antworte auf Deutsch.
""",
    mcp_servers=[autoshop_mcp],
)


### 2. Externer Enrichment-Agent

Dieser Agent hat **keinen MCP-Zugriff**. Er darf nur allgemeine Zusatzinformationen und Verkaufsargumente formulieren.


In [ ]:
external_enrichment_agent = Agent(
    name="External Car Enrichment Agent",
    model=AI_MODEL,
    instructions="""
Du bist eine externe KI ohne Zugriff auf den AutoShop-Catalog und ohne MCP-Zugriff.

Du darfst:
- allgemeine Zusatzinformationen liefern,
- mögliche sinnvolle Extras vorschlagen,
- typische Vorteile eines Fahrzeugtyps erklären,
- allgemeine Verkaufsargumente formulieren.

Du darfst nicht:
- Preise erfinden,
- Shop-Verfügbarkeit erfinden,
- interne IDs erfinden,
- neue konkrete Shop-Fahrzeuge erfinden.

Kennzeichne Extras als unverbindliche Vorschläge.
Antworte auf Deutsch.
""",
)


### 3. Offert-Agent

Der Offert-Agent erhält nur die bereits ermittelten Resultate. Er besitzt ebenfalls keinen MCP-Zugriff und darf interne Fakten nicht verändern.


In [ ]:
offer_agent = Agent(
    name="AutoShop Offer Agent",
    model=AI_MODEL,
    instructions="""
Du erstellst eine Demo-Offerte als Markdown.

Regeln:
- Fahrzeuge, IDs und Preise dürfen ausschliesslich aus den übergebenen internen AutoShop-Daten stammen.
- Externe Informationen dürfen nur als unverbindliche Zusatzinformationen verwendet werden.
- Erfinde keine Preise, IDs, Verfügbarkeiten oder Fahrzeuge.
- Die Offerte ist ein Education-Beispiel und nicht rechtsverbindlich.
- Antworte auf Deutsch.
""",
)


---

## Vollständiger Multi-Agent-Ablauf

Der MCP-Server wird für den gesamten Workflow einmal verbunden. Der `Runner` übernimmt bei jedem Agenten den Modell-/Tool-Loop.


In [ ]:
async def car_sales_agent(customer_name: str, user_request: str):
    async with autoshop_mcp:
        print("1. Interner Agent sucht AutoShop-Daten über MCP ...")
        internal_run = await Runner.run(
            internal_car_agent,
            f"""
Kundenwunsch:
{user_request}

Suche passende Fahrzeuge im AutoShop und gib eine begründete Empfehlung ab.
""",
        )
        internal_result = internal_run.final_output

        print("2. Externer Agent ergänzt allgemeine Zusatzinformationen ...")
        external_run = await Runner.run(
            external_enrichment_agent,
            f"""
Kundenwunsch:
{user_request}

Interne AutoShop-Daten:
{internal_result}

Ergänze:
- mögliche sinnvolle Extras,
- typische Vorteile dieses Fahrzeugtyps,
- Punkte, auf die der Kunde achten sollte,
- kurze Verkaufsargumente.
""",
        )
        external_result = external_run.final_output

        print("3. Offert-Agent erstellt die Markdown-Offerte ...")
        offer_run = await Runner.run(
            offer_agent,
            f"""
Erstelle eine Markdown-Offerte.

Kunde:
{customer_name}

Kundenwunsch:
{user_request}

Interne AutoShop-Daten:
{internal_result}

Externe Zusatzinformationen:
{external_result}

Verwende exakt diese Struktur:

# Offerte

## Kunde

## Kundenwunsch

## Empfohlenes Fahrzeug

## Preis gemäss AutoShop-Catalog

## Empfohlene Extras

## Begründung

## Hinweis

Der Hinweis muss enthalten:
Diese Offerte ist ein Education-Beispiel und nicht rechtsverbindlich.
""",
        )

    return {
        "internal_result": internal_result,
        "external_result": external_result,
        "offer": offer_run.final_output,
    }


---

## Aufruf


In [ ]:
from IPython.display import Markdown, display

result = await car_sales_agent(
    customer_name="Max Muster",
    user_request="Ich suche ein günstiges Auto mit viel Platz für die Familie.",
)

display(Markdown(result["offer"]))


### Optional: Zwischenresultate anzeigen


In [ ]:
display(Markdown("## Internes Agentenresultat"))
display(Markdown(result["internal_result"]))

display(Markdown("## Externes Enrichment"))
display(Markdown(result["external_result"]))


---

## Was sich gegenüber der ursprünglichen Variante geändert hat

| Vorher | Agents SDK |
|---|---|
| `OpenAI(...).responses.create()` direkt | `Agent(...)` + `Runner.run(...)` |
| MCP-Tools selber mit `session.list_tools()` laden | `MCPServerStreamableHttp` |
| MCP-Schemas selber in OpenAI-Tools konvertieren | automatische MCP-Integration |
| `function_call` selber auswerten | Agent Runner übernimmt den Tool-Loop |
| `session.call_tool()` selber aufrufen | Agent Runner ruft MCP-Tools auf |
| eigener `max_rounds`-Loop | SDK verwaltet Agenten-Turns |
| eigene Read-only-Auswahl | `create_static_tool_filter(...)` |

Die fachliche Trennung bleibt erhalten:

**Kundenwunsch → interner MCP-Agent → externer Enrichment-Agent → Offert-Agent → Markdown-Offerte**


## Hinweise

- `order_delete` wird durch die Allowlist nicht an den internen Agenten exponiert.
- Der externe Agent und der Offert-Agent erhalten keinen MCP-Server.
- Bei direkter Nutzung der OpenAI Platform kann SDK-Tracing wieder aktiviert werden.
- Falls dein `AI_BASE_URL` kein Responses-kompatibles OpenAI API bereitstellt, muss der Modellprovider entsprechend auf Chat Completions umgestellt werden. Das Notebook geht wie die bisherige Version davon aus, dass dein Endpoint die OpenAI Responses API unterstützt.
